## ⚙️ Configurazione

In [1]:
# --- Parametri da modificare ---

DATASET   = "EMP/Yeast_H3"          # nome relativo sotto DATA_ROOT
FM_MODEL  = "InstaDeepAI/NTv3_650M_pre"  # HuggingFace model ID
K         = 4                        # ordine dei k-mer (feature dim = 4^K)

# --- Percorsi (raramente da toccare) ---
DATA_ROOT  = "/data/genomic_bench/dna_foundation_benchmark/"
CACHE_DIR  = "../cache/fm_embeddings"
FM_BATCH_SIZE = 64   # sequenze per batch durante l'inferenza FM
N_WORKERS     = 8    # workers per l'estrazione FCGR

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import discover_datasets, load_dataset
from src.fm_experiment.fm_embedder import FMEmbedder
from src.fm_experiment.kmer_features import extract_kmer_features
from src.fm_experiment.ridge_mapping import fit_and_evaluate
from src.fm_experiment.records import (
    load_records,
    has_kmer, has_fm, has_ridge,
    write_kmer, write_fm, write_ridge,
    get_kmer, get_fm, get_ridge,
    RECORDS_CSV,
)

print(f"Dataset : {DATASET}")
print(f"FM model: {FM_MODEL}")
print(f"k-mer k : {K}  →  {4**K} features")

/home/oem/miniconda3/envs/cgr_bench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset : EMP/Yeast_H3
FM model: InstaDeepAI/NTv3_650M_pre
k-mer k : 4  →  256 features


## 1. Caricamento sequenze

In [3]:
all_datasets = discover_datasets(DATA_ROOT)
ds = next(d for d in all_datasets if d["name"] == DATASET)

train_seqs, train_labels, test_seqs, test_labels = load_dataset(
    ds["train_path"], ds["test_path"]
)

print(f"Train: {len(train_seqs):,} sequenze")
print(f"Test : {len(test_seqs):,} sequenze")
print(f"Lunghezza esempio: {len(train_seqs[0])} bp")
print(f"Classi: {len(set(train_labels))}")

Train: 10,475 sequenze
Test : 4,490 sequenze
Lunghezza esempio: 500 bp
Classi: 2


## 2. Embedding FM  
Carica dalla cache se già presente, altrimenti esegue l'inferenza su GPU e salva.

In [4]:
fm = FMEmbedder(model_name=FM_MODEL, cache_dir=CACHE_DIR)

Loading model InstaDeepAI/NTv3_650M_pre on cuda ...


Loading weights: 100%|██████████| 297/297 [00:00<00:00, 14286.14it/s]


Model ready.


In [5]:
safe_name = DATASET.replace("/", "__")
train_cache = os.path.join(CACHE_DIR, safe_name, "train.npz")
test_cache  = os.path.join(CACHE_DIR, safe_name, "test.npz")

if os.path.exists(train_cache) and os.path.exists(test_cache):
    print("✓ Cache trovata — skip inferenza FM")
else:
    print("Cache assente — avvio inferenza FM (potrebbe richiedere qualche minuto)...")

Y_train = fm.embed_sequences(train_seqs, DATASET, "train", FM_BATCH_SIZE)
Y_test  = fm.embed_sequences(test_seqs,  DATASET, "test",  FM_BATCH_SIZE)

print(f"\nEmbedding train : {Y_train.shape}  dtype={Y_train.dtype}")
print(f"Embedding test  : {Y_test.shape}   dtype={Y_test.dtype}")
print(f"Cache salvata in: {train_cache}")

✓ Cache trovata — skip inferenza FM

Embedding train : (10475, 1536)  dtype=float16
Embedding test  : (4490, 1536)   dtype=float16
Cache salvata in: ../cache/fm_embeddings/EMP__Yeast_H3/train.npz


## 3. Feature k-mer da FCGR

In [6]:
X_train = extract_kmer_features(train_seqs, k=K, grid_size=128, n_workers=N_WORKERS)
X_test  = extract_kmer_features(test_seqs,  k=K, grid_size=128, n_workers=N_WORKERS)

print(f"K-mer train : {X_train.shape}  dtype={X_train.dtype}")
print(f"K-mer test  : {X_test.shape}   dtype={X_test.dtype}")

FCGR: 100%|██████████| 4490/4490 [00:00<00:00, 13870.44it/s]


K-mer train : (10475, 256)  dtype=float32
K-mer test  : (4490, 256)   dtype=float32


## 4. Ridge Regression

In [7]:
records = load_records(os.path.join("..", RECORDS_CSV))

if has_ridge(DATASET, K, FM_MODEL, records):
    ridge_res = get_ridge(DATASET, K, FM_MODEL, records)
    print("✓ Ridge già in records — skip riaddestrament")
else:
    ridge_raw = fit_and_evaluate(
        X_train,
        Y_train.astype(np.float32),
        X_test,
        Y_test.astype(np.float32),
    )
    ridge_res = {"R2": ridge_raw["r2_global"], "MSE": ridge_raw["mse_global"]}
    # salva anche r2_per_dim per l'analisi più sotto
    _ridge_r2_per_dim   = ridge_raw["r2_per_dim"]
    _ridge_r2_median    = ridge_raw["r2_median_dim"]
    _ridge_best_alpha   = ridge_raw["best_alpha"]
    _ridge_n_embed_dims = ridge_raw["n_embed_dims"]
    write_ridge(DATASET, K, FM_MODEL, ridge_res,
                path=os.path.join("..", RECORDS_CSV))
    records = load_records(os.path.join("..", RECORDS_CSV))
    print(f"Salvato in {RECORDS_CSV}")

print(f"\nRidge  R²  : {ridge_res['R2']:.4f}")
print(f"Ridge  MSE : {ridge_res['MSE']:.4f}")

✓ Ridge già in records — skip riaddestrament

Ridge  R²  : 0.3697
Ridge  MSE : 99.7866


## 5. Analisi

In [8]:
# r2_per_dim è disponibile solo se Ridge è stato addestrato in questa sessione
if "_ridge_r2_per_dim" not in dir():
    print("ℹ️  r2_per_dim non disponibile (dati caricati da records) — skip grafici Ridge per dimensione")
else:
    r2_per_dim = _ridge_r2_per_dim
    n_dims = len(r2_per_dim)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"{DATASET}  |  FM: {FM_MODEL.split('/')[-1]}  |  k={K}", fontsize=11)

    ax = axes[0]
    ax.hist(r2_per_dim, bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
    ax.axvline(ridge_res["R2"],    color="tomato", lw=1.8, ls="--", label=f"media {ridge_res['R2']:.3f}")
    ax.axvline(_ridge_r2_median,   color="gold",   lw=1.8, ls=":",  label=f"mediana {_ridge_r2_median:.3f}")
    ax.set_xlabel("R² per dimensione")
    ax.set_ylabel("# dimensioni")
    ax.set_title("Distribuzione R² per dimensione embedding")
    ax.legend(fontsize=9)

    ax = axes[1]
    sorted_r2 = np.sort(r2_per_dim)[::-1]
    ax.fill_between(range(n_dims), sorted_r2, alpha=0.6, color="steelblue")
    ax.axhline(ridge_res["R2"],   color="tomato", lw=1.5, ls="--", label=f"media {ridge_res['R2']:.3f}")
    ax.axhline(_ridge_r2_median,  color="gold",   lw=1.5, ls=":",  label=f"mediana {_ridge_r2_median:.3f}")
    ax.set_xlabel("Dimensione (ordinata per R² decrescente)")
    ax.set_ylabel("R²")
    ax.set_title("R² per dimensione — ordinato")
    ax.legend(fontsize=9)
    ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.show()

    frac_above_05 = (r2_per_dim > 0.5).mean()
    frac_below_0  = (r2_per_dim < 0.0).mean()
    print(f"Dimensioni con R² > 0.5 : {frac_above_05*100:.1f}%  ({int(frac_above_05*n_dims)}/{n_dims})")
    print(f"Dimensioni con R² < 0   : {frac_below_0*100:.1f}%  ({int(frac_below_0*n_dims)}/{n_dims})")

ℹ️  r2_per_dim non disponibile (dati caricati da records) — skip grafici Ridge per dimensione


## 6. Random Forest su feature k-mer

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import matthews_corrcoef, roc_auc_score, f1_score, accuracy_score

_RF_PARAM_GRID = {
    "n_estimators":      [200, 500, 1000],
    "max_features":      ["sqrt", "log2"],
    "max_depth":         [20, None],
    "min_samples_split": [2, 5],
}
n_classes = len(set(train_labels))
scoring   = "roc_auc" if n_classes == 2 else "accuracy"
cv        = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

def _train_rf(X_tr, y_tr):
    gs = GridSearchCV(
        RandomForestClassifier(n_jobs=4, random_state=42),
        _RF_PARAM_GRID, scoring=scoring, cv=cv, n_jobs=1, refit=True,
    )
    gs.fit(X_tr, y_tr)
    return gs

def _eval_rf(model, X_te, y_te):
    y_pred = model.predict(X_te)
    mcc = matthews_corrcoef(y_te, y_pred)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="macro")
    if n_classes == 2:
        auroc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    else:
        auroc = roc_auc_score(y_te, model.predict_proba(X_te),
                              multi_class="ovr", average="macro")
    return {"MCC": mcc, "AUROC": auroc, "F1": f1, "Accuracy": acc}

# --- k-mer RF ---
records = load_records(os.path.join("..", RECORDS_CSV))

if has_kmer(DATASET, K, records):
    res_kmer = get_kmer(DATASET, K, records)
    print(f"✓ RF k-mer k={K} già in records — skip riaddestrament")
else:
    print(f"Training RF su k-mer (k={K}, {X_train.shape[1]} dim)...")
    rf_kmer  = _train_rf(X_train, train_labels)
    res_kmer = _eval_rf(rf_kmer, X_test, test_labels)
    write_kmer(DATASET, K, res_kmer, path=os.path.join("..", RECORDS_CSV))
    records = load_records(os.path.join("..", RECORDS_CSV))
    print(f"Salvato in {RECORDS_CSV}")

print("\n── RF su k-mer features ──")
for m, v in res_kmer.items():
    print(f"  {m:10s}: {float(v):.4f}")

✓ RF k-mer k=4 già in records — skip riaddestrament

── RF su k-mer features ──
  MCC       : 0.6551
  AUROC     : 0.9120
  F1        : 0.8273
  Accuracy  : 0.8276


## 7. Random Forest su embedding FM

In [ ]:
records = load_records(os.path.join("..", RECORDS_CSV))

if has_fm(DATASET, FM_MODEL, records):
    res_fm = get_fm(DATASET, FM_MODEL, records)
    print(f"✓ RF FM già in records — skip riaddestrament")
else:
    print(f"Training RF su FM embeddings ({Y_train.shape[1]} dim)...")
    rf_fm  = _train_rf(Y_train.astype(np.float32), train_labels)
    res_fm = _eval_rf(rf_fm, Y_test.astype(np.float32), test_labels)
    write_fm(DATASET, FM_MODEL, res_fm, path=os.path.join("..", RECORDS_CSV))
    records = load_records(os.path.join("..", RECORDS_CSV))
    print(f"Salvato in {RECORDS_CSV}")

print("\n── RF su FM embeddings ──")
for m, v in res_fm.items():
    print(f"  {m:10s}: {float(v):.4f}")

Training RF su FM embeddings (1536 dim)...


## 8. Confronto k-mer vs FM

In [ ]:
metrics_names = ["MCC", "AUROC", "F1", "Accuracy"]
vals_kmer = [float(res_kmer[m]) for m in metrics_names]
vals_fm   = [float(res_fm[m])   for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars_k = ax.bar(x - width/2, vals_kmer, width, label=f"RF k-mer (k={K})", color="steelblue")
bars_f = ax.bar(x + width/2, vals_fm,   width, label="RF FM embeddings",  color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title(f"{DATASET}  —  RF: k-mer vs FM embeddings")
ax.legend()
ax.bar_label(bars_k, fmt="%.3f", padding=2, fontsize=8)
ax.bar_label(bars_f, fmt="%.3f", padding=2, fontsize=8)
ax.axhline(1.0, color="gray", lw=0.5, ls="--")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({"k-mer": vals_kmer, "FM embeddings": vals_fm}, index=metrics_names)
summary["Δ (FM − k-mer)"] = summary["FM embeddings"] - summary["k-mer"]
print(summary.round(4).to_string())

## 9. Records aggiornati

In [ ]:
records_display = load_records(os.path.join("..", RECORDS_CSV))
print(f"Records path: results/records.csv  ({len(records_display)} dataset/i)\n")
records_display